# 01 · Can the GM Trust the Model's Win Probabilities?

**Audience:** Carolina Panthers general manager (hypothetical stakeholder)

**Question:** When our Elo model says a team has an X% chance to win, does that team actually win about X% of the time? And are the model's season-level playoff odds just as trustworthy?

**Context:** The GM uses our weekly game probabilities and rest-of-season playoff odds. Before acting on them, the GM needs to know whether a "70%" from the model means 70%.

**Assumptions & scope:** Elo with the tuned `DEFAULT_PARAMS` (K=21, HFA=42, regression 0.4). Game-level checks use the held-out 2022–2025 seasons (regular season and playoffs), which the parameters were never tuned on. Playoff-odds checks use projections made after Week 7 of 2021–2025. Everything is frozen at 2026 Week 7 to match the existing write-up.

**Methods:**
1. **Brier scores:** the same train/test split and functions as `notebooks/03_elo_baseline.ipynb`, imported from `nflmodel`.
2. **Game-level calibration table:** out-of-sample home win probabilities grouped into 10-point bins, with the actual home win rate in each bin. The same table is built from the betting market (no-vig moneyline probabilities from the same games table) for comparison.
3. **Playoff-odds calibration:** the 06 backtest re-run with the same imported `project_season`, settings, and random seed.

Supporting context for Carolina's own outlook is in `carolina_outlook_gm.ipynb`. Locked settings are in `definitions.py`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from nflmodel.data import read_games
from nflmodel.elo import run_elo, DEFAULT_PARAMS, team_history
from nflmodel.evaluate import brier_score, brier_baselines
from nflmodel.projections import project_season, team_wins

from definitions import (FIRST_SEASON, LAST_SEASON, TRAIN_SEASONS, TEST_SEASONS,
                         PROB_BINS, PROB_LABELS, SEASON, THROUGH_WEEK, BACKTEST_SEASONS,
                         N_SIMS, PLAYOFF_ODDS_BINS, PLAYOFF_ODDS_LABELS)

TABLES = Path("outputs/tables")
TABLES.mkdir(parents=True, exist_ok=True)

## 1. Brier scores (reproduces 03 Section 5)

Brier score = average squared error of the probabilities (lower is better; a coin flip scores 0.25). Skill score (BSS) = improvement over a coin flip.

In [2]:
# Same filtering and parameters as 03_elo_baseline
games = read_games()
games_03 = games[games["season"].between(FIRST_SEASON, LAST_SEASON)].reset_index(drop=True)
elo = run_elo(games_03, DEFAULT_PARAMS)

rows = []
for name, seasons in [("train 2013-2021", TRAIN_SEASONS), ("test 2022-2025", TEST_SEASONS)]:
    df = elo[elo["season"].isin(list(seasons))]
    b = brier_score(df)
    base = brier_baselines(df)
    rows.append({"split": name,
                 "games": int(df["home_win"].notna().sum()),
                 "brier_elo": b,
                 "brier_coin_flip": base["coin_flip"],
                 "brier_home_rate": base["home_rate"],
                 "bss_vs_coin_flip": 1 - b / base["coin_flip"]})

brier_table = pd.DataFrame(rows)
brier_table.to_csv(TABLES / "brier_scores_train_test.csv", index=False)
brier_table.round(4)

,split,games,brier_elo,brier_coin_flip,brier_home_rate,bss_vs_coin_flip
0,train 2013-2021,2423,0.2188,0.25,0.2469,0.1249
1,test 2022-2025,1139,0.2212,0.25,0.2471,0.1152


## 2. Game-level calibration table (out-of-sample, 2022–2025)

Market probability = moneyline implied probabilities for both teams, rescaled to sum to 1 (vig removed). Each forecaster is binned by its *own* probability, so the two halves of the table can have different game counts per bin.

In [3]:
def implied_prob(moneyline):
    """Implied win probability from an American moneyline (includes the vig)."""
    return np.where(moneyline < 0, -moneyline / (-moneyline + 100), 100 / (moneyline + 100))

# Out-of-sample games with a result (ties count as half a home win, as in the Brier score)
test = elo[elo["season"].isin(list(TEST_SEASONS)) & elo["home_win"].notna()].copy()

home_raw = implied_prob(test["home_moneyline"])
away_raw = implied_prob(test["away_moneyline"])
test["market_prob"] = home_raw / (home_raw + away_raw)
print(f"Out-of-sample games: {len(test)}  |  missing moneylines: {test['market_prob'].isna().sum()}")


def calibration_table(df, prob_col, source):
    """Games, average predicted probability, and actual home win rate per 10-point bin."""
    d = df[df[prob_col].notna()].copy()
    d["bin"] = pd.cut(d[prob_col], PROB_BINS, labels=PROB_LABELS, include_lowest=True)
    t = (d.groupby("bin", observed=False)
          .agg(games=("home_win", "size"),
               avg_predicted=(prob_col, "mean"),
               actual_win_rate=("home_win", "mean"))
          .reset_index())
    t["gap"] = t["actual_win_rate"] - t["avg_predicted"]   # + means the home team won more than predicted
    t.insert(0, "source", source)
    return t

calib = pd.concat([calibration_table(test, "home_prob", "Elo"),
                   calibration_table(test, "market_prob", "Market (no-vig)")], ignore_index=True)
calib.to_csv(TABLES / "game_calibration_by_bin_2022_2025.csv", index=False)
calib.round(3)

Out-of-sample games: 1139  |  missing moneylines: 0


,source,bin,games,avg_predicted,actual_win_rate,gap
0,Elo,0-10%,0,NaN,NaN,NaN
1,Elo,10-20%,19,0.157,0.211,0.053
2,Elo,20-30%,63,0.262,0.381,0.119
3,Elo,30-40%,129,0.360,0.364,0.004
4,Elo,40-50%,210,0.450,0.407,-0.043
5,Elo,50-60%,254,0.552,0.512,-0.041
6,Elo,60-70%,216,0.650,0.694,0.045
7,Elo,70-80%,180,0.746,0.728,-0.018
8,Elo,80-90%,59,0.835,0.847,0.012
9,Elo,90-100%,9,0.909,1.000,0.091


In [4]:
# Head-to-head on the same games: Elo vs market Brier score
both = test[test["market_prob"].notna()]
head_to_head = pd.DataFrame({
    "forecaster": ["Elo", "Market (no-vig)"],
    "games": len(both),
    "brier": [((both["home_prob"] - both["home_win"]) ** 2).mean(),
              ((both["market_prob"] - both["home_win"]) ** 2).mean()],
    # Count-weighted average absolute gap between predicted and actual across bins
    "avg_abs_calibration_gap": [
        np.average(calib.loc[calib["source"] == s, "gap"].abs().fillna(0),
                   weights=calib.loc[calib["source"] == s, "games"])
        for s in ["Elo", "Market (no-vig)"]],
})
head_to_head.to_csv(TABLES / "elo_vs_market_brier_2022_2025.csv", index=False)
head_to_head.round(4)

,forecaster,games,brier,avg_abs_calibration_gap
0,Elo,1139,0.2212,0.0376
1,Market (no-vig),1139,0.2095,0.0228


## 3. Playoff-odds calibration (reproduces 06 Section 4)

Projections made after Week 7 in each backtest season, compared with who actually made the playoffs. Same function, settings, and seed as 06.

In [5]:
# Elo on the full games table, exactly as 06 runs it
elo_all = run_elo(games, DEFAULT_PARAMS)
history = team_history(elo_all)

rows = []
for s in BACKTEST_SEASONS:
    p, _, _ = project_season(games, history, s, THROUGH_WEEK, DEFAULT_PARAMS, N_SIMS)
    season_games = games[games["season"] == s]
    final = team_wins(season_games[season_games["game_type"] == "REG"])
    postseason = season_games[season_games["is_playoff"]]

    p["actual_wins"] = final["sum"].values
    p["made_playoffs"] = p.index.isin(set(postseason["home_team"]) | set(postseason["away_team"]))
    p["season"] = s
    rows.append(p.reset_index())

backtest = pd.concat(rows, ignore_index=True)
backtest["error"] = backtest["proj_wins"] - backtest["actual_wins"]
backtest.to_csv(TABLES / f"backtest_thru_wk{THROUGH_WEEK:02d}_2021_2025.csv", index=False)
RMSE = np.sqrt((backtest["error"] ** 2).mean())

backtest["odds_bin"] = pd.cut(backtest["playoff_pct"], PLAYOFF_ODDS_BINS, labels=PLAYOFF_ODDS_LABELS)
playoff_calib = (backtest.groupby("odds_bin", observed=True)
                 .agg(teams=("team", "size"),
                      avg_projected=("playoff_pct", "mean"),
                      actually_made=("made_playoffs", "mean"))
                 .reset_index())
playoff_calib.to_csv(TABLES / "playoff_odds_calibration_thru_wk07.csv", index=False)

print(f"Backtest RMSE: {RMSE:.2f} wins ({len(backtest)} team-seasons)")
playoff_calib.round(2)

Backtest RMSE: 1.96 wins (160 team-seasons)


,odds_bin,teams,avg_projected,actually_made
0,Under 10%,47,0.03,0.06
1,10-50%,48,0.29,0.33
2,50-90%,38,0.75,0.74
3,Over 90%,27,0.96,0.85


## 4. Notes for the GM

The tables above are saved to `outputs/tables/`. The plain-language interpretation lives in this folder's write-up; the key caution is that calibration is checked on 2022–2025 games and 2021–2025 playoff races, so a bin with few games (see the `games` and `teams` columns) is weak evidence on its own.